<a href="https://colab.research.google.com/github/nemo10-boop/WUEKM/blob/main/Quantum_AlexNet_Model_FMNIST_Noise10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Package Installation

In [ ]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 62.4 MB/s eta 0:00:00


Library Imports & Device Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import pennylane as qml
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


Quantum Circuit & Variational Layer Definition

In [ ]:
import pennylane as qml

n_qubits = 8
dev = qml.device("lightning.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="adjoint")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    num_layers = weights.shape[0]
    for layer in range(num_layers):
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i + 1) % n_qubits])
        for i in range(n_qubits):
            qml.RZ(weights[layer, i], wires=i)

    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

# INCREASE DEPTH: Change from 4 layers to 8 layers
weight_shapes = {"weights": (8, n_qubits)}
quantum_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)


Data Pipeline (FashionMNIST Loaders & Noise Injection)(10.0 %)

In [ ]:
def get_fmnist_loaders(noise_percentage=0.0, batch_size=64):

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    def add_noise(dataset, percentage):
        if percentage == 0.0:
            return dataset

        noise_factor = percentage / 100.0
        data = dataset.data.float() / 255.0          # [N, 28, 28]
        noise = torch.randn_like(data) * noise_factor
        noisy_data = torch.clamp(data + noise, 0.0, 1.0)
        dataset.data = (noisy_data * 255).to(torch.uint8)
        return dataset

    train_dataset = add_noise(train_dataset, noise_percentage)
    test_dataset  = add_noise(test_dataset,  noise_percentage)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, test_loader

# Configure your noise setting here (0.0, 5.0, 10.0)
NOISE_LEVEL = 10.0
train_loader, test_loader = get_fmnist_loaders(noise_percentage=NOISE_LEVEL, batch_size=64)

100%|██████████| 26.4M/26.4M [00:01<00:00, 16.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 270kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.99MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 18.1MB/s]


Hybrid Quantum-AlexNet Architecture Construction

In [ ]:
class HybridQuantumAlexNet(nn.Module):
    def __init__(self, q_layer, n_qubits):
        super(HybridQuantumAlexNet, self).__init__()

        alexnet = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        self.features = alexnet.features

        for param in self.features.parameters():
            param.requires_grad = False

        self.avgpool = alexnet.avgpool
        self.flatten = nn.Flatten()

        # FIXED: Gradual, expressive classical compression pipeline
        self.dense1 = nn.Linear(9216, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.act1 = nn.LeakyReLU(0.1)

        self.dense2 = nn.Linear(512, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.act2 = nn.LeakyReLU(0.1)

        self.linear_reduction = nn.Linear(128, n_qubits)

        self.quantum_layer = q_layer
        self.classifier = nn.Linear(n_qubits, 10)

    def forward(self, x):
        x = x.to(device)

        # 1. Classical Feature Extraction
        x = self.features(x)
        x = self.avgpool(x)
        x = self.flatten(x)

        # 2. Gradual expressive transition layers
        x = self.dense1(x)
        x = self.bn1(x)
        x = self.act1(x)

        x = self.dense2(x)
        x = self.bn2(x)
        x = self.act2(x)

        x = self.linear_reduction(x)
        x = torch.sin(x) * (np.pi / 2)

        # 3. Quantum State Mapping
        quantum_outputs = [self.quantum_layer(item) for item in x]
        x = torch.stack(quantum_outputs)

        # 4. Final Classification
        x = self.classifier(x)
        return x

model = HybridQuantumAlexNet(quantum_layer, n_qubits).to(device)
print("model architecture compiled.")

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:02<00:00, 86.5MB/s]


model architecture compiled.


Training / Evaluation Subsets & Optimizer Hyperparameters

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset



# Scale training subset up to 1,500 images to stabilize batch metrics
cpu_subset_indices = list(range(1500))
train_loader_cpu = DataLoader(
    Subset(train_loader.dataset, cpu_subset_indices),
    batch_size=32,
    shuffle=True,
    # num_workers=2,  # Easy to uncomment if needed
    # pin_memory=True
)

# Scale testing subset up to 500 images
cpu_test_subset_indices = list(range(500))
test_loader_cpu = DataLoader(
    Subset(test_loader.dataset, cpu_test_subset_indices),
    batch_size=32,
    shuffle=False  # No need to shuffle when evaluating test performance
)


# Define loss function
criterion = nn.CrossEntropyLoss()

# Lower learning rate to 0.001 to prevent optimization overshooting
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

print("Learning rate safely throttled to 0.001.")

Learning rate safely throttled to 0.001.


Helper Functions for Model Training and Testing & Model Execution Switchboard (Training and Evaluation)

In [ ]:
import torch

def train_model(model, train_loader, criterion, optimizer, epochs=3):
    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            # Move data to target device (CPU/GPU)
            images, labels = images.to(device), labels.to(device)

            # Standard PyTorch optimization step
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Metrics calculation
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (
                predicted
                .eq(labels)
                .sum()
                .item()
            )

            # Frequent batch logging
            if batch_idx % 50 == 0:
                print(
                    f"Epoch [{epoch+1}/{epochs}] "
                    f"Batch [{batch_idx}/{len(train_loader)}] "
                    f"Loss: {loss.item():.4f} "
                    f"Accuracy: {100. * correct / total:.2f}%"
                )

        # End of epoch summary
        print(
            f"==> Epoch {epoch+1} Complete. "
            f"Loss: {running_loss / len(train_loader):.4f}, "
            f"Accuracy: {100. * correct / total:.2f}%\n"
        )

def test_model(model, test_loader, criterion):
    model.eval()

    test_loss = 0
    correct = 0
    total = 0

    # Disable gradient tracking for evaluation
    with torch.no_grad():
        for images, labels in test_loader:
            # Move data to target device
            images, labels = images.to(device), labels.to(device)

            # Forward pass & loss extraction
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Metrics calculation
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (
                predicted
                .eq(labels)
                .sum()
                .item()
            )

    # Final evaluation summary
    print(
        f"\n[Test Results] "
        f"Loss: {test_loss / len(test_loader):.4f}, "
        f"Accuracy: {100. * correct / total:.2f}%\n"
    )


# Run on the sampled CPU training subset:
train_model(model, train_loader_cpu, criterion, optimizer, epochs=8)


# --- Testing Execution Options ---
# Run on the sampled CPU testing subset:
test_model(model, test_loader_cpu, criterion)

# Run on the FULL test loader dataset:
# test_model(model, test_loader, criterion)

Epoch [1/8] Batch [0/47] Loss: 2.3949 Accuracy: 9.38%
==> Epoch 1 Complete. Loss: 2.0018, Accuracy: 43.13%

Epoch [2/8] Batch [0/47] Loss: 1.9533 Accuracy: 50.00%
==> Epoch 2 Complete. Loss: 1.8258, Accuracy: 62.07%

Epoch [3/8] Batch [0/47] Loss: 1.7586 Accuracy: 62.50%
==> Epoch 3 Complete. Loss: 1.7146, Accuracy: 71.27%

Epoch [4/8] Batch [0/47] Loss: 1.6053 Accuracy: 75.00%
==> Epoch 4 Complete. Loss: 1.6146, Accuracy: 76.20%

Epoch [5/8] Batch [0/47] Loss: 1.4886 Accuracy: 87.50%
==> Epoch 5 Complete. Loss: 1.5325, Accuracy: 78.00%

Epoch [6/8] Batch [0/47] Loss: 1.4404 Accuracy: 87.50%
==> Epoch 6 Complete. Loss: 1.4532, Accuracy: 79.20%

Epoch [7/8] Batch [0/47] Loss: 1.3434 Accuracy: 81.25%
==> Epoch 7 Complete. Loss: 1.3583, Accuracy: 83.40%

Epoch [8/8] Batch [0/47] Loss: 1.2267 Accuracy: 90.62%
==> Epoch 8 Complete. Loss: 1.2908, Accuracy: 86.53%


[Test Results] Loss: 1.4559, Accuracy: 72.40%



In [ ]:
# ─── Mount Google Drive & Save Model ───────────────────────────────────────────
from google.colab import drive
import os

# Mount Drive (opens auth prompt on first run)
drive.mount('/content/drive')

# ── Define save path ──
SAVE_DIR = '/content/drive/MyDrive/QuantumAlexNet_model_checkpoints_Noise10'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Save model weights ──
MODEL_PATH = os.path.join(SAVE_DIR, 'model_weights.pth')
torch.save(model.state_dict(), MODEL_PATH)
print(f"✅ Model weights saved to: {MODEL_PATH}")

# ── Optional: save full checkpoint (weights + optimizer state + epoch) ──
CHECKPOINT_PATH = os.path.join(SAVE_DIR, 'checkpoint.pth')
torch.save({
    'model_state_dict':     model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch':                8,   # update to match your last epoch
}, CHECKPOINT_PATH)
print(f"✅ Full checkpoint saved to: {CHECKPOINT_PATH}")

Mounted at /content/drive
✅ Model weights saved to: /content/drive/MyDrive/QuantumAlexNet_model_checkpoints_Noise10/model_weights.pth
✅ Full checkpoint saved to: /content/drive/MyDrive/QuantumAlexNet_model_checkpoints_Noise10/checkpoint.pth
